# Sanity check: do the SQL metrics match an independent calculation?

The dashboard's numbers come from SQL views and functions. This notebook recomputes them in pandas from the **raw tables** (`observations`, `forecasts`), without using any view or `rpc_*` function, and compares the two. They must agree to 3 decimal places.

The window is the frozen report window, Wed 4 Nov 00:00 to Sat 7 Nov 00:00 MYT.

In [1]:
import numpy as np
import pandas as pd

from iww_ingest import config, db

FROM = pd.Timestamp('2026-11-03T16:00:00Z')   # Wed 4 Nov 00:00 MYT
TO = pd.Timestamp('2026-11-06T16:00:00Z')     # Sat 7 Nov 00:00 MYT
conn = db.connect(config.settings().db_url, 'sanity')


def frame(query, params=()):
    with conn.cursor() as cur:
        cur.execute(query, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])

## 1. Rebuild the hourly actuals from raw 15-minute samples

Hour H covers (H−1 h, H]. An hour only counts when all 4 samples exist. Temperature is the sample at H itself, and rain is the sum of the four samples.

In [2]:
obs = frame("""select observed_at, temperature_c, precipitation_mm from observations
                where source = 'openmeteo_current' and observed_at > %s and observed_at <= %s""",
            (FROM - pd.Timedelta(hours=1), TO))
obs['observed_at'] = pd.to_datetime(obs['observed_at'], utc=True)
obs['hour'] = obs['observed_at'].dt.ceil('h')

hourly = obs.groupby('hour').agg(samples=('observed_at', 'size'), rain_mm=('precipitation_mm', 'sum'))
hourly['temperature_c'] = obs[obs['observed_at'] == obs['hour']].set_index('hour')['temperature_c']
hourly = hourly[(hourly['samples'] == 4) & (hourly.index < TO)]
hourly['rain_hour'] = hourly['rain_mm'] >= 0.1
print(f"{len(hourly)} complete hours, {int(hourly['rain_hour'].sum())} rain hours")
hourly.head(3)

72 complete hours, 35 rain hours


,samples,rain_mm,temperature_c,rain_hour
hour,,,,
2026-11-03 16:00:00+00:00,4,0.0,25.7,False
2026-11-03 17:00:00+00:00,4,0.4,25.8,True
2026-11-03 18:00:00+00:00,4,0.8,25.9,True


## 2. Pair each stored forecast with its actual

Every lead is loaded, and the leaderboard later keeps only 24–144 h, the range all four models share. `best_match+ml` comes from `predictions`, the corrected values the model wrote.

In [3]:
fc = frame("""select model, target_time, lead_hours, temperature_c, precipitation_mm from forecasts
               where target_time >= %s and target_time < %s""", (FROM, TO))
ml = frame("""select target_time, lead_hours, value as temperature_c from predictions
               where variable = 'temperature_c' and target_time >= %s and target_time < %s""", (FROM, TO)).assign(model='best_match+ml', precipitation_mm=np.nan)
fc = pd.concat([fc, ml], ignore_index=True)
fc['target_time'] = pd.to_datetime(fc['target_time'], utc=True)
pairs = fc.merge(hourly, left_on='target_time', right_index=True, suffixes=('', '_obs'))
pairs['error'] = pairs['temperature_c'] - pairs['temperature_c_obs']
print(f"{len(pairs):,} pairs")

2,448 pairs


## 3. Temperature MAE and bias per model and lead bucket

In [4]:
bucket = pd.cut(pairs['lead_hours'], [24, 48, 72, 120, 169], right=False, labels=['024-048', '048-072', '072-120', '120-168'])
mine = pairs.groupby(['model', bucket], observed=True)['error'].agg(
    n='size', mae=lambda e: e.abs().mean(), bias='mean').reset_index()

sql = frame("""select model, lead_bucket, n, mae, bias from rpc_metrics_by_lead(%s, %s)
                where variable = 'temperature_c'""", (FROM, TO)).rename(columns={'lead_bucket': 'lead_hours'})
sql['lead_hours'] = sql['lead_hours'].astype(str)
mine['lead_hours'] = mine['lead_hours'].astype(str)
compared = mine.merge(sql, on=['model', 'lead_hours'], suffixes=('_pandas', '_sql'))
compared['mae_diff'] = (compared['mae_pandas'] - compared['mae_sql']).abs()
compared['bias_diff'] = (compared['bias_pandas'] - compared['bias_sql']).abs()
assert (compared['n_pandas'] == compared['n_sql']).all()
assert len(compared) == 20
assert compared[['mae_diff', 'bias_diff']].max().max() < 0.0005
compared[['model', 'lead_hours', 'n_pandas', 'n_sql', 'mae_pandas', 'mae_sql', 'bias_pandas', 'bias_sql']].round(3)

,model,lead_hours,n_pandas,n_sql,mae_pandas,mae_sql,bias_pandas,bias_sql
0,best_match,024-048,72,72,1.207,1.207,-1.049,-1.049
1,best_match,048-072,72,72,1.169,1.169,-1.031,-1.031
2,best_match,072-120,144,144,1.324,1.324,-1.109,-1.109
3,best_match,120-168,216,216,1.426,1.426,-1.093,-1.093
4,best_match+ml,024-048,72,72,0.915,0.915,-0.027,-0.027
5,best_match+ml,048-072,72,72,0.794,0.794,-0.006,-0.006
6,best_match+ml,072-120,144,144,0.921,0.921,-0.074,-0.074
7,best_match+ml,120-168,216,216,0.928,0.928,-0.017,-0.017
8,ecmwf_ifs025,024-048,72,72,1.217,1.217,-1.072,-1.072
9,ecmwf_ifs025,048-072,72,72,1.165,1.165,-1.110,-1.110


## 4. Leaderboard temperature MAE and rain CSI

In [5]:
rows = []
for model, g in pairs[pairs['lead_hours'] <= 144].groupby('model'):
    rain = g.dropna(subset=['precipitation_mm'])
    hit = ((rain['precipitation_mm'] >= 0.1) & rain['rain_hour']).sum()
    any_rain = ((rain['precipitation_mm'] >= 0.1) | rain['rain_hour']).sum()
    rows.append({'model': model, 'temp_mae_pandas': g['error'].abs().mean(),
                 'csi_pandas': hit / any_rain if any_rain else np.nan})
board = pd.DataFrame(rows).merge(
    frame('select model, temp_mae as temp_mae_sql, rain_csi as csi_sql from rpc_leaderboard(%s, %s)', (FROM, TO)), on='model')
for a, b in (('temp_mae_pandas', 'temp_mae_sql'), ('csi_pandas', 'csi_sql')):
    both = board[[a, b]].astype(float).dropna()
    assert (both[a] - both[b]).abs().max() < 0.0005, a
board.round(3)

,model,temp_mae_pandas,csi_pandas,temp_mae_sql,csi_sql
0,best_match,1.296,0.482,1.296,0.482
1,best_match+ml,0.874,NaN,0.874,NaN
2,ecmwf_ifs025,1.388,0.483,1.388,0.483
3,gfs_seamless,0.827,0.397,0.827,0.397
4,icon_seamless,1.062,0.444,1.062,0.444


## 5. Rain contingency table, 1 day ahead, Best Match

In [6]:
day1 = pairs[(pairs['model'] == 'best_match') & (pairs['lead_hours'] == 24)]
fc_rain, obs_rain = day1['precipitation_mm'] >= 0.1, day1['rain_hour']
a, b, c, d = (fc_rain & obs_rain).sum(), (fc_rain & ~obs_rain).sum(), (~fc_rain & obs_rain).sum(), (~fc_rain & ~obs_rain).sum()
sql = frame("""select hits, false_alarms, misses, correct_negatives from rpc_rain_contingency(%s, %s)
                where model = 'best_match' and lead_bucket = '024-048'""", (FROM, TO)).iloc[0]
assert (a, b, c, d) == tuple(sql)
print(f'hits {a}, false alarms {b}, misses {c}, correct negatives {d}: identical in pandas and SQL')
print(f'POD {a / (a + c):.3f}, FAR {b / (a + b):.3f}, CSI {a / (a + b + c):.3f}')

hits 24, false alarms 19, misses 11, correct negatives 18: identical in pandas and SQL
POD 0.686, FAR 0.442, CSI 0.444


## 6. Does the ML model improve on the raw forecast?

In [7]:
shared = pairs[pairs['lead_hours'] <= 144]
raw = shared[shared['model'] == 'best_match']['error']
cor = shared[shared['model'] == 'best_match+ml']['error']
print(f'Raw Best Match:  MAE {raw.abs().mean():.3f} °C, bias {raw.mean():+.3f} °C, n = {len(raw)}')
print(f'With ML:         MAE {cor.abs().mean():.3f} °C, bias {cor.mean():+.3f} °C, n = {len(cor)}')
print(f'MAE change:      {100 * (cor.abs().mean() / raw.abs().mean() - 1):+.1f} %')
print()
print('All checks passed: SQL and pandas agree to 3 decimal places.')
conn.close()

Raw Best Match:  MAE 1.296 °C, bias -1.084 °C, n = 432
With ML:         MAE 0.874 °C, bias -0.043 °C, n = 432
MAE change:      -32.6 %

All checks passed: SQL and pandas agree to 3 decimal places.
